# Act 4 & 5 · Semantic Property Graph (`GRAPH_EXPAND`), Intersection Consent & Measured Cost Model

Resolving records into a `person_id` is only the halfway mark of a Composable CDP. To activate customer data safely and profitably, we need:
1. **Declarative Semantic Property Graph (`cdp.cdp_semantic_graph`):** Using BigQuery's native `MEASURE()` DDL (`70_graph.sql`) and `GRAPH_EXPAND` + `AGG()`, analysts and **BigQuery Conversational Analytics Data Agents** can aggregate multi-hop metrics (`Profile -> Membership -> Identifier`) without risking fan-out double-counting.
2. **Intersection Consent under the Australian Privacy Act 1988 (`85_consent.sql`):** Under APP 7, an explicit `WITHDRAWN` opt-out on any source record for a `(person, channel, purpose)` must override a `GRANTED` opt-in on another record for that same person.
3. **Downstream Activation & Household Mail De-Duplication (`90_downstream.sql`):** Joining loyalty spend across all resolved records per customer and eliminating duplicate postal mailings within the same household (`cdp.golden_household`).
4. **Measured Cost Model (`96_cost_model.sql`):** Auditing actual token consumption and BigQuery compute from `INFORMATION_SCHEMA.JOBS` and extrapolating to 15M records.

In [1]:
%load_ext bigquery_magics

import bigquery_magics
import pandas as pd
from IPython.display import display
from pathlib import Path

# ---------------------------------------------------------------------------
# Set PROJECT if running outside a checkout that has demo/config.env
PROJECT = ""          # e.g. "all-things-cdp"
DATASET = ""          # leave blank for the default, "cdp"
# ---------------------------------------------------------------------------

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 52)

CFG, CONFIG, _source = {}, None, None

def _find_config():
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        for candidate in (base / "config.env", base / "demo" / "config.env"):
            if candidate.is_file():
                return candidate
    return None

if not PROJECT:
    CONFIG = _find_config()
    if CONFIG:
        for _line in CONFIG.read_text().splitlines():
            _line = _line.strip()
            if _line and not _line.startswith("#") and "=" in _line:
                _k, _v = _line.split("=", 1)
                CFG[_k.strip()] = _v.split("#", 1)[0].strip().strip('"').strip("'")
        PROJECT = CFG.get("CDP_PROJECT", "")
        _source = str(CONFIG)

if not PROJECT:
    import os
    PROJECT = os.environ.get("GOOGLE_CLOUD_PROJECT") or os.environ.get("GCP_PROJECT") or "all-things-cdp"
    _source = "environment / default"

DATASET = DATASET or CFG.get("CDP_DS") or CFG.get("CDP_DATASET_PREFIX") or "cdp"
LOCATION = CFG.get("CDP_LOCATION", "US")

bigquery_magics.context.project = PROJECT
bigquery_magics.context.location = LOCATION
bigquery_magics.context.progress_bar_type = None

print(f"project    {PROJECT}   (from {_source})")
print(f"dataset    {DATASET}   (location {LOCATION})")

project    all-things-cdp   (from demo/config.env)
dataset    cdp   (location US)

## 1 · Declarative Graph Measures with `GRAPH_EXPAND` + `AGG()` (`cdp.cdp_semantic_graph`)

In traditional SQL, joining `Profile (20,000)` $\rightarrow$ `Membership (28,906)` $\rightarrow$ `Identifier (18,724)` fans out profile-level metrics by the number of identifiers per profile.
In `70_graph.sql`, `cdp.cdp_semantic_graph` declares node-scoped measures (`MEASURE(COUNT(1)) AS profile_count`, `MEASURE(COUNT(DISTINCT baseline_wesid)) AS baseline_wesid_count`, etc.). When queried via `FROM GRAPH_EXPAND("cdp.cdp_semantic_graph")`, BigQuery's `AGG()` operator automatically evaluates each measure at its declared grain!

In [2]:
%%bigquery df_semantic_graph
SELECT
  Profile_sources AS brand,
  AGG(Profile_profile_count)                    AS profiles,
  AGG(Membership_link_count)                    AS links,
  AGG(Identifier_identifier_count)              AS distinct_identifiers,
  AGG(Profile_profiles_touching_hubs)           AS touching_hubs,
  AGG(Profile_baseline_wesid_count)             AS baseline_clusters,
  AGG(Profile_weighted_wesid_count)             AS weighted_clusters,
  AGG(Profile_composable_wesid_count)           AS composable_clusters,
  AGG(Profile_profiles_in_baseline_hairballs)   AS in_baseline_hairballs,
  AGG(Profile_profiles_in_weighted_hairballs)   AS in_weighted_hairballs,
  AGG(Profile_profiles_in_composable_hairballs) AS in_composable_hairballs
FROM GRAPH_EXPAND("cdp.cdp_semantic_graph")
GROUP BY brand
ORDER BY profiles DESC

brand,profiles,links,distinct_identifiers,touching_hubs,baseline_clusters,weighted_clusters,composable_clusters,in_baseline_hairballs,in_weighted_hairballs,in_composable_hairballs
CRM,4155,9085,8295,679,2948,4133,4154,1196,0,0
ECOM,3752,6110,5386,628,2733,3730,3751,1008,0,0
LOYALTY,3248,7095,6503,525,2287,3248,3246,962,0,0
POS,2516,2956,1665,927,1189,2255,2252,1180,0,0
ENRICH,1842,2165,1711,291,1268,1704,1718,486,0,0
SUPPORT,854,1148,979,132,568,823,819,265,0,0
CALL,257,347,277,66,160,257,257,98,0,0


## 2 · Intersection Consent Governance (`cdp.v_consent_impact` & `cdp.v_consent_conflicts`)

When a customer has 3 source records — two saying `GRANTED` for marketing email and one saying `WITHDRAWN` — a naive **Union Rule** (`ANY(GRANTED)`) illegally contacts a customer who explicitly opted out!
Stage 85 (`85_consent.sql`) enforces **Intersection Consent**: a single `WITHDRAWN` assertion across any resolved record in the cluster suppresses marketing on that channel (`actual_decision = 'WITHDRAWN'`), and any person carrying a `RISK_FLAG` is automatically suppressed across all marketing channels.

In [3]:
%%bigquery df_consent
SELECT
  channel,
  purpose,
  people_with_a_stated_preference,
  audience_union_rule,
  audience_intersection_rule,
  contacts_the_union_rule_would_add,
  of_which_explicitly_withdrawn,
  suppressed_for_risk,
  pct_of_union_audience_unlawful
FROM `cdp.v_consent_impact`
WHERE purpose = 'MARKETING'
ORDER BY people_with_a_stated_preference DESC

channel,purpose,people_with_a_stated_preference,audience_union_rule,audience_intersection_rule,contacts_the_union_rule_would_add,of_which_explicitly_withdrawn,suppressed_for_risk,pct_of_union_audience_unlawful
EMAIL,MARKETING,3351,2063,1861,202,72,66,9.79
SMS,MARKETING,2271,1378,1286,92,32,66,6.68
POST,MARKETING,1697,1000,951,49,17,66,4.9
PHONE,MARKETING,1270,756,733,23,6,66,3.04


In [4]:
%%bigquery df_consent_conflicts
SELECT
  person_id,
  full_name,
  channel,
  granted_count,
  withdrawn_count,
  would_have_been_contacted,
  actual_decision,
  explanation
FROM `cdp.v_consent_conflicts`
LIMIT 5

person_id,full_name,channel,granted_count,withdrawn_count,would_have_been_contacted,actual_decision,explanation
PER-4490c5b70d037280,Lucille Osman,POST,2,1,GRANTED,WITHDRAWN,Withdrawn on 2019-05-21. Withdrawal is absolute and is not overridden by a later grant from another source.
PER-52c80bb634a27ec5,Tengfei Chi,EMAIL,2,1,GRANTED,WITHDRAWN,Withdrawn on 2023-10-19. Withdrawal is absolute and is not overridden by a later grant from another source.
PER-79474a4aaffe6762,Malti Nagpal,EMAIL,2,1,GRANTED,WITHDRAWN,Withdrawn on 2021-07-15. Withdrawal is absolute and is not overridden by a later grant from another source.
PER-44486f5a5ef42466,Areej Hamade,EMAIL,1,1,GRANTED,WITHDRAWN,Withdrawn on 2025-10-10. Withdrawal is absolute and is not overridden by a later grant from another source.
PER-4e9c97bed29db7a7,Jackson Larson,EMAIL,1,1,GRANTED,WITHDRAWN,Withdrawn on 2025-10-14. Withdrawal is absolute and is not overridden by a later grant from another source.


## 3 · Downstream Activation Lift & Household Direct-Mail Savings (`90_downstream.sql`)

Let's inspect the two commercial ROI views built in Stage 90:
1. **`cdp.v_activation_before_after`:** Connecting POS and ECOM transactions across all resolved records increases attributed customer reach from **1,658 loyalty cards (`$162,366` spend)** to **5,293 resolved customers (`$479,989` spend)** — nearly **3x attributed revenue**.
2. **`cdp.v_household_waste`:** Grouping resolved people by `golden_household` (`address_key`) eliminates **12.85% of postal direct mailings** (`111` duplicate catalogue mailings avoided across `753` mailable households, including `30` multi-surname households).

In [5]:
%%bigquery df_roi
SELECT * FROM `cdp.v_activation_before_after`

basis,customers,total_spend,spend_per_customer
resolved · one person = one customer,5293,479989.09,90.68
unresolved · one loyalty card = one customer,1658,162366,97.93


In [6]:
%%bigquery df_waste
SELECT * FROM `cdp.v_household_waste`

households,people,duplicate_mailings_avoided,pct_of_mailings_avoidable,multi_person_households,multi_surname_households
753,864,111,12.85,100,30


## 4 · Measured AI & Compute Cost Model (`cdp.v_cost_model` & `cdp.v_cost_compute`)

99.61% of candidate pairs are settled deterministically by the 2-Hop Graph + Hybrid Search tiering rules. Only the `0.39%` grey zone (41,769 pairs) reaches Gemini — but at ~1,570 input + ~90 output tokens per judgement, **that is where almost all the money goes**.

`cdp.v_cost_model` prices **one clean rebuild** of the 20,000-record corpus at **`$131.59`** of model spend (`$131.51` adjudication; extraction + embedding ≈ `$0.09`), i.e. **`$6.58` per 1,000 records** and a linear floor of **`~$98.7k` per full rebuild at 15M records**. BigQuery compute (`cdp.v_cost_compute`) is extra.

> **How this is measured:** the adjudicator's token usage is now captured from `AI.GENERATE`'s `usage_metadata`. Ledger rows written before that fix are costed at the measured per-call average of a 300-call calibration sample, so the `basis` column reads `ESTIMATED`. Cumulative adjudication spend across all iterations of this demo (re-runs included) is reported separately as `usd_adjudication_ledger_to_date`.
>
> **What this means at scale:** a full nightly rebuild at 15M records is not how you would run this. In steady state only *new or changed* records go through the grey zone, and tightening `tau_low` / `tau_high` trades recall for cost (`cdp.v_cost_sensitivity`).

In [7]:
%%bigquery df_cost
SELECT
  demo_records,
  demo_people_resolved,
  ai_calls,
  total_tokens,
  usd_extraction,
  usd_embedding,
  usd_adjudication,
  usd_model_total,
  usd_model_per_1k_records,
  target_records,
  usd_model_at_target_per_rebuild,
  usd_adjudication_ledger_to_date
FROM `cdp.v_cost_model`

demo_records,demo_people_resolved,ai_calls,total_tokens,usd_extraction,usd_embedding,usd_adjudication,usd_model_total,usd_model_per_1k_records,target_records,usd_model_at_target_per_rebuild,usd_adjudication_ledger_to_date
20000,13566,64147,69712806,0.0367,0.0483,131.5064,131.5914,6.5796,15000000,98693.56,362.17
